In [17]:
%pip install sympy z3-solver

Note: you may need to restart the kernel to use updated packages.


In [18]:
from dataclasses import dataclass

import sympy as sp
import z3

def S(name):
    """Все символические числа в этой тетради по умолчанию целые."""
    return sp.Symbol(name, integer=True)

@dataclass(frozen=True)
class Vector2:
    x: sp.Expr
    y: sp.Expr

    def __add__(self, other: "Vector2") -> "Vector2":
        return Vector2(self.x + other.x, self.y + other.y)

    def __sub__(self, other: "Vector2") -> "Vector2":
        return Vector2(self.x - other.x, self.y - other.y)

    def __repr__(self) -> str:
        return f"({self.x},{self.y})"


def D(n, k, points: set[Vector2]) -> frozenset[Vector2]:
    return frozenset(
        point + Vector2(i * a, j * b)
        for point in points
        for a, b in ((k, n), (n, k))
        for i in (-1, 1)
        for j in (-1, 1)
    )

In [19]:
n = S("n")
k = S("k")
origin = Vector2(0, 0)


def D1(points: set[Vector2]) -> frozenset[Vector2]:
    return D(n, k, points)


def count_intersections(left, right):
    """For every center p, count (p + left) intersect right."""
    counts = {}
    for left_point in left:
        for right_point in right:
            center = right_point - left_point
            counts[center] = counts.get(center, 0) + 1
    return counts


def refine_relations(initial_relations, max_rounds=4):
    relations = []
    known_relations = set()

    for relation in initial_relations:
        relation = frozenset(relation)
        if relation and relation not in known_relations:
            relations.append(relation)
            known_relations.add(relation)

    new_by_round = []

    for _ in range(max_rounds):
        count_maps = [
            count_intersections(left, right)
            for left in relations
            for right in relations
        ]
        existing_points = set().union(*relations)
        intersection_centers = set().union(*(counts.keys() for counts in count_maps))
        candidates = existing_points | intersection_centers
        grouped = {}

        for point in candidates:
            membership = tuple(point in relation for relation in relations)
            intersection_counts = tuple(counts.get(point, 0) for counts in count_maps)
            signature = membership + intersection_counts
            grouped.setdefault(signature, set()).add(point)

        new_relations = []
        for signature in sorted(grouped):
            relation = frozenset(grouped[signature])
            if relation not in known_relations:
                new_relations.append(relation)
                known_relations.add(relation)

        new_by_round.append(tuple(new_relations))
        if not new_relations:
            break
        relations.extend(new_relations)

    return tuple(relations), tuple(new_by_round)


@dataclass(frozen=True)
class DClass:
    a: sp.Expr
    b: sp.Expr

    def __repr__(self) -> str:
        return f"D({self.a},{self.b})"


def unsigned(expression):
    expression = sp.expand(expression)
    choices = (expression, sp.expand(-expression))
    return min(
        choices,
        key=lambda value: (str(value).lstrip().startswith("-"), sp.default_sort_key(value)),
    )


def as_d_class(point):
    coordinates = sorted(
        (unsigned(point.x), unsigned(point.y)),
        key=sp.default_sort_key,
        reverse=True,
    )
    return DClass(*coordinates)


def clump_d(relation):
    return tuple(sorted({as_d_class(point) for point in relation}, key=repr))


MAX_ROUNDS = 1
MAX_D_CLASSES_PER_GROUP = 5

base_relation = D1({origin})
relations, new_by_round = refine_relations([base_relation], max_rounds=MAX_ROUNDS)

all_discovered_relations = (base_relation,) + tuple(
    relation
    for round_relations in new_by_round
    for relation in round_relations
)

clumped_groups = {clump_d(relation) for relation in all_discovered_relations}


def is_zero_class(d_class):
    return sp.simplify(d_class.a) == 0 and sp.simplify(d_class.b) == 0


def is_integer_multiple_of_base(d_class):
    for base_a, base_b in ((n, k), (k, n)):
        multiplier = sp.simplify(d_class.a / base_a)
        is_positive_integer = (
            not multiplier.free_symbols
            and multiplier.is_integer is True
            and multiplier.is_positive is True
        )
        if is_positive_integer and sp.simplify(d_class.b - multiplier * base_b) == 0:
            return True
    return False


def is_filtered_singleton(group):
    if len(group) != 1:
        return False
    d_class = group[0]
    return is_zero_class(d_class) or is_integer_multiple_of_base(d_class)



def group_scale(group):
    coefficients = []
    for d_class in group:
        for expression in (d_class.a, d_class.b):
            expression = sp.expand(expression)
            coefficient_n = expression.coeff(n)
            coefficient_k = expression.coeff(k)
            remainder = sp.simplify(expression - coefficient_n * n - coefficient_k * k)
            if remainder != 0:
                return None
            coefficients.extend((coefficient_n, coefficient_k))

    nonzero = [abs(coefficient) for coefficient in coefficients if coefficient != 0]
    return min(nonzero) if nonzero else sp.Integer(1)


def proportional_group_key(group):
    scale = group_scale(group)
    if scale is None:
        return group

    normalized = {
        as_d_class(Vector2(sp.simplify(d_class.a / scale), sp.simplify(d_class.b / scale)))
        for d_class in group
    }
    return tuple(sorted(normalized, key=repr))


candidate_groups = [
    group
    for group in clumped_groups
    if len(group) <= MAX_D_CLASSES_PER_GROUP and not is_filtered_singleton(group)
]
candidate_groups.sort(
    key=lambda group: (
        float(group_scale(group)) if group_scale(group) is not None else float("inf"),
        len(group),
        tuple(map(repr, group)),
    )
)

representatives_by_proportion = {}
for group in candidate_groups:
    representatives_by_proportion.setdefault(proportional_group_key(group), group)

filtered_groups = tuple(
    sorted(
        representatives_by_proportion.values(),
        key=lambda group: (len(group), tuple(map(repr, group))),
    )
)


def format_groups(groups):
    lines = []
    for group_number, group in enumerate(groups, start=1):
        lines.append(f"group_{group_number}:")
        for d_class in group:
            lines.append(f"\t{d_class}")

    return "\n".join(lines) if lines else "(no matching groups)"


print(format_groups(filtered_groups))


group_1:
	D(2*k,0)
	D(2*n,0)
	D(k + n,k + n)
	D(k + n,k - n)
	D(k - n,k - n)


In [20]:
def pm1(a, b):
    return sp.Or(sp.Eq(a + 1, b), sp.Eq(a - 1, b))

def is_neighbour(a1, b1, A1, B1, a2, b2, A2, B2):
    return sp.Or(
            sp.And(
                pm1(a1, a2), 
                pm1(B1, B2), 
                sp.Eq(A1, A2), 
                sp.Eq(b1, b2)
            ), 
            sp.And(
                pm1(A1, A2), 
                pm1(b1, b2), 
                sp.Eq(a1, a2), 
                sp.Eq(B1, B2)
            )
        )

def level(a, b, A, B):
    return sp.Max(sp.Abs(a), sp.Abs(B)) + sp.Max(sp.Abs(b), sp.Abs(A))


In [ ]:
a1, b1, A1, B1 = [S(i) for i in ["a1", "b1", "A1", "B1"]]
a2, b2, A2, B2 = [S(i) for i in ["a2", "b2", "A2", "B2"]]
a3, b3, A3, B3 = [S(i) for i in ["a3", "b3", "A3", "B3"]]
acom, bcom, Acom, Bcom = [S(i) for i in ["a_com", "b_com", "A_com", "B_com"]]

equation_1 = sp.And(
    # Points are neighbours
    is_neighbour(a1, b1, A1, B1, acom, bcom, Acom, Bcom), 
    is_neighbour(a2, b2, A2, B2, acom, bcom, Acom, Bcom), 
    # Points are not equal
    sp.Or(
        sp.Ne(a1, a2),
        sp.Ne(b1, b2),
        sp.Ne(A1, A2),
        sp.Ne(B1, B2),
    ),
    # Level of points is equal or deeper, than current
    sp.And(
        level(a1, b1, A1, B1) <= level(acom, bcom, Acom, Bcom),
        level(a2, b2, A2, B2) <= level(acom, bcom, Acom, Bcom)
    ),
    # The amount of steps done is the same for x and y
    sp.And(
        sp.Eq(sp.Mod(sp.Abs(a1) + sp.Abs(b1), 2), sp.Mod(sp.Abs(A1) + sp.Abs(B1), 2)),
        sp.Eq(sp.Mod(sp.Abs(a2) + sp.Abs(b2), 2), sp.Mod(sp.Abs(A2) + sp.Abs(B2), 2)),
        sp.Eq(sp.Mod(sp.Abs(acom) + sp.Abs(bcom), 2), sp.Mod(sp.Abs(Acom) + sp.Abs(Bcom), 2)),
    ),
    # Poinnts are accessible
    sp.And(
        sp.Eq(sp.Mod(a1 - B1, 2), 0),
        sp.Eq(sp.Mod(b1 - A1, 2), 0),
        sp.Eq(sp.Mod(a2 - B2, 2), 0),
        sp.Eq(sp.Mod(b2 - A2, 2), 0),
        sp.Eq(sp.Mod(acom - Bcom, 2), 0),
        sp.Eq(sp.Mod(bcom - Acom, 2), 0)
    )
)

equation = equation_1

In [22]:
@dataclass(frozen=True)
class ConditionalCandidate:
    point: tuple
    condition: object


class UnsupportedFiniteEquation(ValueError):
    pass


def _candidate_order(expression, preferred_symbols):
    """Выбирает более удобную, но равносильную запись точки."""
    foreign_symbols = expression.free_symbols - preferred_symbols
    return (
        len(foreign_symbols),
        sp.count_ops(expression),
        tuple(sorted(map(str, expression.free_symbols))),
        sp.srepr(expression),
    )


def solve_equation_symbolically(equation, unknowns, preferred_symbols=()):
    """
    Находит конечные символические кандидаты прямо из equation.

    Для каждого кандидата condition точно говорит, при каких значениях
    параметров этот кандидат является решением.
    """
    unknowns = tuple(unknowns)
    unknown_set = set(unknowns)
    preferred_symbols = set(preferred_symbols)

    if not unknowns:
        raise ValueError("Нужно указать хотя бы одну неизвестную")

    # Условия только на параметры не нужно размножать при раскрытии скобок.
    top_level_parts = sp.And.make_args(equation)
    dependent_parts = tuple(
        part for part in top_level_parts
        if part.free_symbols & unknown_set
    )
    if not dependent_parts:
        raise UnsupportedFiniteEquation(
            "Формула не ограничивает неизвестные: решений бесконечно много"
        )

    dependent_formula = sp.And(*dependent_parts)
    dnf = sp.to_dnf(dependent_formula, simplify=False, force=True)
    branches = sp.Or.make_args(dnf)
    point_forms = set()

    for branch in branches:
        if branch is sp.false:
            continue

        atoms = sp.And.make_args(branch)
        assignment = {}

        for unknown in unknowns:
            possible_values = set()
            for atom in atoms:
                if not isinstance(atom, sp.Equality):
                    continue
                if unknown not in atom.free_symbols:
                    continue

                # Abs, Mod, Max и неравенства только фильтруют уже
                # найденные точки. Кандидаты строятся из обычных
                # линейных равенств, например a_com = a1 + 1.
                difference = atom.lhs - atom.rhs
                if not difference.is_polynomial(unknown):
                    continue

                for value in sp.solve(atom, unknown):
                    # Значение должно быть выражено только через параметры.
                    if not (value.free_symbols & unknown_set):
                        possible_values.add(value)

            if not possible_values:
                raise UnsupportedFiniteEquation(
                    f"В одной из ветвей {unknown} не определяется "
                    "отдельным равенством. Возможно, решений бесконечно много."
                )

            assignment[unknown] = min(
                possible_values,
                key=lambda value: _candidate_order(value, preferred_symbols),
            )

        point_forms.add(tuple(assignment[unknown] for unknown in unknowns))

    result = []
    for point in sorted(point_forms, key=str):
        substitution = dict(zip(unknowns, point))
        condition = equation.subs(substitution, simultaneous=True)
        if condition is not sp.false:
            result.append(ConditionalCandidate(point, condition))

    return tuple(result)


@dataclass(frozen=True)
class SolutionSetCase:
    points: frozenset
    condition: object
    basis_points: tuple


def _sympy_to_z3(expression, symbols):
    if expression is sp.true:
        return z3.BoolVal(True)
    if expression is sp.false:
        return z3.BoolVal(False)
    if isinstance(expression, sp.Symbol):
        return symbols[expression]
    if isinstance(expression, sp.Integer):
        return z3.IntVal(int(expression))
    if isinstance(expression, sp.Rational):
        return z3.RealVal(f"{expression.p}/{expression.q}")
    if isinstance(expression, sp.Add):
        return sum(
            (_sympy_to_z3(arg, symbols) for arg in expression.args),
            z3.IntVal(0),
        )
    if isinstance(expression, sp.Mul):
        result = z3.IntVal(1)
        for arg in expression.args:
            result *= _sympy_to_z3(arg, symbols)
        return result
    if isinstance(expression, sp.Pow):
        base, exponent = expression.args
        if not exponent.is_Integer or exponent < 0:
            raise UnsupportedFiniteEquation(f"Не поддерживается степень {expression}")
        return _sympy_to_z3(base, symbols) ** int(exponent)
    if expression.func == sp.Abs:
        value = _sympy_to_z3(expression.args[0], symbols)
        return z3.If(value >= 0, value, -value)
    if expression.func in (sp.Max, sp.Min):
        values = [_sympy_to_z3(arg, symbols) for arg in expression.args]
        result = values[0]
        for value in values[1:]:
            if expression.func == sp.Max:
                result = z3.If(result >= value, result, value)
            else:
                result = z3.If(result <= value, result, value)
        return result
    if expression.func == sp.Mod:
        value, modulus = expression.args
        return _sympy_to_z3(value, symbols) % _sympy_to_z3(modulus, symbols)
    if isinstance(expression, sp.Equality):
        return _sympy_to_z3(expression.lhs, symbols) == _sympy_to_z3(expression.rhs, symbols)
    if isinstance(expression, sp.Unequality):
        return _sympy_to_z3(expression.lhs, symbols) != _sympy_to_z3(expression.rhs, symbols)
    if isinstance(expression, sp.StrictLessThan):
        return _sympy_to_z3(expression.lhs, symbols) < _sympy_to_z3(expression.rhs, symbols)
    if isinstance(expression, sp.LessThan):
        return _sympy_to_z3(expression.lhs, symbols) <= _sympy_to_z3(expression.rhs, symbols)
    if isinstance(expression, sp.StrictGreaterThan):
        return _sympy_to_z3(expression.lhs, symbols) > _sympy_to_z3(expression.rhs, symbols)
    if isinstance(expression, sp.GreaterThan):
        return _sympy_to_z3(expression.lhs, symbols) >= _sympy_to_z3(expression.rhs, symbols)
    if isinstance(expression, sp.And):
        return z3.And(*(_sympy_to_z3(arg, symbols) for arg in expression.args))
    if isinstance(expression, sp.Or):
        return z3.Or(*(_sympy_to_z3(arg, symbols) for arg in expression.args))
    if isinstance(expression, sp.Not):
        return z3.Not(_sympy_to_z3(expression.args[0], symbols))

    raise UnsupportedFiniteEquation(
        f"Проверка возможных множеств пока не поддерживает {expression.func}"
    )


def find_all_solution_sets(candidates, basis_point_symbols=()):
    """Возвращает все совместно достижимые множества кандидатов."""
    candidates = tuple(candidates)
    if not candidates:
        return (SolutionSetCase(frozenset(), sp.true, tuple()),)

    parameter_symbols = set().union(*(
        candidate.condition.free_symbols for candidate in candidates
    ))
    z3_symbols = {symbol: z3.Int(str(symbol)) for symbol in parameter_symbols}
    solver = z3.Solver()
    flags = []
    z3_conditions = []

    for number, candidate in enumerate(candidates):
        flag = z3.Bool(f"__candidate_is_solution_{number}")
        flags.append(flag)
        z3_condition = _sympy_to_z3(candidate.condition, z3_symbols)
        z3_conditions.append(z3_condition)
        solver.add(flag == z3_condition)

    cases = []
    while solver.check() == z3.sat:
        model = solver.model()
        pattern = tuple(
            z3.is_true(model.eval(flag, model_completion=True))
            for flag in flags
        )
        points = frozenset(
            candidate.point
            for candidate, is_solution in zip(candidates, pattern)
            if is_solution
        )
        condition = sp.And(*(
            candidate.condition if is_solution else sp.Not(candidate.condition)
            for candidate, is_solution in zip(candidates, pattern)
        ))

        # Ищем небольшой числовой пример для этого точного набора.
        optimizer = z3.Optimize()
        for z3_condition, is_solution in zip(z3_conditions, pattern):
            optimizer.add(z3_condition if is_solution else z3.Not(z3_condition))
        distance_from_zero = sum(
            (z3.If(value >= 0, value, -value) for value in z3_symbols.values()),
            z3.IntVal(0),
        )
        optimizer.minimize(distance_from_zero)
        if optimizer.check() != z3.sat:
            raise RuntimeError("Не удалось восстановить уже найденный пример")
        small_model = optimizer.model()
        basis_points = tuple(
            tuple(
                small_model.eval(z3_symbols[symbol], model_completion=True).as_long()
                for symbol in point_symbols
            )
            for point_symbols in basis_point_symbols
        )
        cases.append(SolutionSetCase(points, condition, basis_points))

        # Запрещаем уже найденный набор истинных/ложных кандидатов.
        solver.add(z3.Or(*(
            flag != z3.BoolVal(is_solution)
            for flag, is_solution in zip(flags, pattern)
        )))

    return tuple(sorted(cases, key=lambda case: (len(case.points), str(case.points))))


candidates = solve_equation_symbolically(
    equation,
    unknowns=(acom, bcom, Acom, Bcom),
    preferred_symbols=(a1, b1, A1, B1),
)

solution_sets = find_all_solution_sets(
    candidates,
    basis_point_symbols=(
        (a1, b1, A1, B1),
        (a2, b2, A2, B2),
    ),
)
print("possible solution counts:", sorted({len(case.points) for case in solution_sets}))
print("number of different solution sets:", len(solution_sets))

for number, case in enumerate(solution_sets, start=1):
    print(f"solution_set_{number}:")
    print("\tcount:", len(case.points))
    print("\tpoints:")
    if not case.points:
        print("\t\tfrozenset()")
    else:
        for point in sorted(case.points, key=str):
            print("\t\t", point)
    print("\tnumerical example:")
    print("\t\tpoint_1:", case.basis_points[0])
    print("\t\tpoint_2:", case.basis_points[1])
    example_values = case.basis_points[0] + case.basis_points[1]
    example_substitution = dict(zip(
        (a1, b1, A1, B1, a2, b2, A2, B2),
        example_values,
    ))
    for solution_number, point in enumerate(sorted(case.points, key=str), start=1):
        numerical_solution = tuple(
            coordinate.subs(example_substitution)
            for coordinate in point
        )
        print(f"\t\tsolution_{solution_number}:", numerical_solution)


possible solution counts: [0, 1]
number of different solution sets: 9
solution_set_1:
	count: 0
	points:
		frozenset()
	numerical example:
		point_1: (0, 0, 0, 0)
		point_2: (0, 0, 0, 0)
solution_set_2:
	count: 1
	points:
		 (a1 + 1, b1, A1, B1 + 1)
	numerical example:
		point_1: (0, 1, -1, 0)
		point_2: (1, 0, 0, 1)
		solution_1: (1, 1, -1, 1)
solution_set_3:
	count: 1
	points:
		 (a1 + 1, b1, A1, B1 - 1)
	numerical example:
		point_1: (0, -1, -1, 0)
		point_2: (1, 0, 0, -1)
		solution_1: (1, -1, -1, -1)
solution_set_4:
	count: 1
	points:
		 (a1 - 1, b1, A1, B1 + 1)
	numerical example:
		point_1: (0, -1, 1, 0)
		point_2: (-1, 0, 0, 1)
		solution_1: (-1, -1, 1, 1)
solution_set_5:
	count: 1
	points:
		 (a1 - 1, b1, A1, B1 - 1)
	numerical example:
		point_1: (0, -1, -1, 0)
		point_2: (-1, 0, 0, -1)
		solution_1: (-1, -1, -1, -1)
solution_set_6:
	count: 1
	points:
		 (a1, b1 + 1, A1 + 1, B1)
	numerical example:
		point_1: (0, 1, -1, 0)
		point_2: (0, 1, 1, 0)
		solution_1: (0, 2, 0, 0)
so